In [ ]:
import itertools
import warnings
import json
import matplotlib.pyplot as plt
import ternary
import pymatgen
from pymatgen.core import Composition
from pymatgen.ext.matproj import MPRester
import smact

# Assuming the SMACT-related code is in a separate file named smact_utils.py
# which includes:
# - smact_filter
# - Oxidation_state_probability_finder
# - smact_validity
# etc.
from smact import Element, element_dictionary
from smact.screening import smact_filter, smact_validity
from smact.oxidation_states import Oxidation_state_probability_finder

###############################################################################
# USER INPUTS
###############################################################################

CHEMICAL_SYSTEM = ["Cu", "Ti", "O"]
MAX_STOICH = 8
MP_API_KEY = "w1FbQcBbkQZJHyhMJGxssIEttJ8JSGu8"  # Replace with your key env version for api safety
MAX_ENERGY_ABOVE_HULL = 0.15  # eV/atom threshold for metastability


In [ ]:

# STEP 1: DEFINE CHEMICAL SPACE

elements = [Element(el) for el in CHEMICAL_SYSTEM]

# Generate all possible compositions up to MAX_STOICH
all_comp_tuples = [
    amt_list for amt_list in itertools.product(range(1, MAX_STOICH+1), repeat=len(elements))
]
all_compositions = [Composition({el.symbol: amt for el, amt in zip(elements, amt_list)})
                    for amt_list in all_comp_tuples]


In [ ]:

# STEP 2: APPLY CHEMICAL FILTERS (SMACT)

# Filter valid compositions using SMACT's validity check
valid_compositions = [comp for comp in all_compositions if smact_validity(comp)]

# Further filter to get full oxidation states & stoichiometries from smact_filter
# Note: smact_filter returns allowed compositions including oxidation states if species_unique=True
filtered_comps_detail = smact_filter(tuple(elements), threshold=MAX_STOICH, species_unique=True)

# From filtered_comps_detail, extract unique reduced compositions
allowed_compositions = []
for entry in filtered_comps_detail:
    syms, ox_states, ratio = entry
    # Build a Composition from ratio & symbols
    c_dict = {s: r for s, r in zip(syms, ratio)}
    comp = Composition(c_dict).reduced_composition
    if comp not in allowed_compositions:
        allowed_compositions.append(comp)


In [ ]:

# STEP 3: OXIDATION STATE PROBABILITY & ADDITIONAL FILTERS

# Initialize the Oxidation state probability finder from SMACT
# (This requires the oxidation_state_probability_table.json data provided by SMACT)
prob_finder = Oxidation_state_probability_finder()

final_allowed_compositions = []
for comp in allowed_compositions:
    # Assign plausible oxidation states and compute a probability score
    # We need species with oxidation states. For simplicity, we assume those 
    # given by smact_filter are acceptable. 
    # Here, we will just use a threshold probability check if desired.
    # As an example, we ignore stoichiometry weighting as in the docstring.
    
    # Convert composition to a list of species (with oxidation states)
    # For demonstration, we skip re-deriving oxidation states and assume composition is feasible.
    # In a real scenario, you'd pick a representative oxidation state combination from filtered_comps_detail.
    
    # If desired, add more stringent filtering based on probability:
    # compound_prob = prob_finder.compound_probability(list_of_species, ignore_stoichiometry=True)
    # if compound_prob > SOME_THRESHOLD:
    #     final_allowed_compositions.append(comp)
    # For simplicity, we accept them all here:
    final_allowed_compositions.append(comp)
    


In [ ]:

# STEP 4: VISUALIZE ON PHASE DIAGRAM (TERNARY PLOT)


# Only works straightforwardly for ternary systems
if len(CHEMICAL_SYSTEM) == 3:
    # Convert each composition into fractional coordinates for ternary plotting
    # Normalize amounts to sum to 1
    plot_points = []
    for comp in final_allowed_compositions:
        fractions = [comp.get_atomic_fraction(el) for el in CHEMICAL_SYSTEM]
        plot_points.append(tuple(fractions))

    fig, ax = plt.subplots()
    figure, tax = ternary.figure(scale=1.0)
    tax.boundary(linewidth=2.0)
    tax.gridlines(color="black", multiple=0.1, linewidth=0.5)
    tax.set_title("Allowed Compositions in {} System".format("-".join(CHEMICAL_SYSTEM)), fontsize=16)
    # Plot allowed compositions
    tax.scatter(plot_points, marker='o', color='green', label='Chemical Filter')

    tax.left_axis_label(CHEMICAL_SYSTEM[0], fontsize=14)
    tax.right_axis_label(CHEMICAL_SYSTEM[1], fontsize=14)
    tax.bottom_axis_label(CHEMICAL_SYSTEM[2], fontsize=14)

    tax.legend()
    tax.show()
else:
    warnings.warn("Visualization code only provided for ternary systems.")


In [ ]:

# STEP 5: CHECKING STABILITY AGAINST MATERIALS PROJECT DATA

# Using pymatgen's MPRester to get phase diagram and hull information.
# This requires internet connection and an MP API key.
with MPRester(MP_API_KEY) as mpr:
    # Get phase diagram entries for the system
    entries = mpr.get_entries_in_chemsys(CHEMICAL_SYSTEM)
    from pymatgen.analysis.phase_diagram import PhaseDiagram
    pd = PhaseDiagram(entries)

    stable_comps = []
    metastable_comps = []

    for comp in final_allowed_compositions:
        # Get energy above hull for the composition
        decomp, e_above_hull = pd.get_decomp_and_e_above_hull(pd.get_entry_by_composition(comp))
        if e_above_hull is not None:
            if e_above_hull <= 0.0:
                stable_comps.append((comp, e_above_hull))
            elif e_above_hull <= MAX_ENERGY_ABOVE_HULL:
                metastable_comps.append((comp, e_above_hull))
    
    print("Stable compositions:", stable_comps)
    print("Metastable compositions (within 0.15 eV/atom):", metastable_comps)


In [ ]:

###############################################################################
# STEP 6: (OPTIONAL) MACE-MP EVALUATION (PLACEHOLDER)
###############################################################################

# Pseudocode:
# for comp in metastable_comps:
#     # Generate initial structure(s) for comp (external code).
#     # Use MACE-MP or similar ML potential to relax structure:
#     # relaxed_structure, energy = run_mace_mp(initial_structure)
#     # If energy improved and still metastable/stable, keep it.
#     pass

###############################################################################
# STEP 7: UPDATE PHASE DIAGRAM WITH NEW DATA
###############################################################################

# After MACE-MP refinement and re-calculation of energies above hull,
# you could re-plot or re-check stability and visualize again.

print("Workflow complete.")
